# Lakebase Search Execution Evidence

Demonstrates retrieval from the **Build 1 Lakebase Search index** (`idx_product_search_bm25`).
No separate vector store is used — all retrieval goes through the Lakebase GIN index on `tsvector`.

Executed: 2026-08-28

In [1]:
import psycopg2, psycopg2.extras, time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
cred = w.postgres.generate_database_credential(
    endpoint="projects/northpeak/branches/dev/endpoints/primary"
)
conn = psycopg2.connect(
    host="ep-calm-band-d2zksq6c.database.us-east-1.cloud.databricks.com",
    database="databricks_postgres",
    user=w.current_user.me().user_name,
    password=cred.token, port=5432, sslmode="require"
)
conn.autocommit = True
print("Connected to Lakebase (dev branch)")

Connected to Lakebase (dev branch)


In [2]:
# Execute BM25 search against idx_product_search_bm25 (Build 1 Lakebase Search index)
search_query = "Summit Down Parka cold weather outerwear"
t0 = time.time()

with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT product_id, product_name, category, subcategory,
               LEFT(description, 120) AS description_preview,
               ts_rank_cd(search_vector,
                   to_tsquery('english', 'summit | down | parka | cold | weather | outerwear')
               ) AS relevance_score
        FROM northpeak_app.product_search
        WHERE search_vector @@ to_tsquery('english', 'summit | down | parka | cold | weather | outerwear')
        ORDER BY relevance_score DESC
        LIMIT 5
    """
    )
    results = cur.fetchall()

elapsed_ms = (time.time() - t0) * 1000
print(f"Index: northpeak_app.idx_product_search_bm25 (GIN on tsvector)")
print(f"Table: northpeak_app.product_search (1998 documents indexed)")
print(f"Query: '{search_query}'")
print(f"Latency: {elapsed_ms:.1f} ms")
print(f"Results: {len(results)} rows")
print("-" * 80)
for i, r in enumerate(results, 1):
    print(f"  [{i}] {r['product_id']:18s} | {r['product_name']:30s} | score={r['relevance_score']:.6f}")
    print(f"      {r['category']} > {r['subcategory']} | {r['description_preview']}")

Index: northpeak_app.idx_product_search_bm25 (GIN on tsvector)
Table: northpeak_app.product_search (1998 documents indexed)
Query: 'Summit Down Parka cold weather outerwear'
Latency: 9.1 ms
Results: 5 rows
--------------------------------------------------------------------------------
  [1] SKU-APP-04412      | Summit Down Parka              | score=0.900000
      Apparel > Outerwear | Heavyweight insulated winter parka, 600-fill down, waterproof shell, storm hood — warmest cold-weather outerwear.
  [2] SKU-APP-10076      | Denim 619                      | score=0.600000
      Apparel > Outerwear | warm cold-weather outerwear denim in apparel.
  [3] SKU-APP-10075      | Polo 229                       | score=0.600000
      Apparel > Outerwear | warm cold-weather outerwear polo in apparel.
  [4] SKU-APP-10025      | Boot 244                       | score=0.600000
      Apparel > Outerwear | warm cold-weather outerwear boot in apparel.
  [5] SKU-APP-10077      | Boot 149                

In [3]:
# Second search — what-if scenario: find substitutes for transfer decision
search_query_2 = "parka jacket insulated winter substitute"
t0 = time.time()

with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT product_id, product_name, category, subcategory,
               LEFT(description, 120) AS description_preview,
               ts_rank_cd(search_vector,
                   to_tsquery('english', 'parka | jacket | insul | winter | substitut')
               ) AS relevance_score
        FROM northpeak_app.product_search
        WHERE search_vector @@ to_tsquery('english', 'parka | jacket | insul | winter | substitut')
        ORDER BY relevance_score DESC
        LIMIT 5
    """
    )
    results_2 = cur.fetchall()

elapsed_ms_2 = (time.time() - t0) * 1000
print(f"Query: '{search_query_2}'")
print(f"Latency: {elapsed_ms_2:.1f} ms")
print(f"Results: {len(results_2)} rows")
print("-" * 80)
for i, r in enumerate(results_2, 1):
    print(f"  [{i}] {r['product_id']:18s} | {r['product_name']:30s} | score={r['relevance_score']:.6f}")
    print(f"      {r['category']} > {r['subcategory']} | {r['description_preview']}")
print("\nAll retrieval via Build 1 Lakebase Search index (no separate vector store).")

Query: 'parka jacket insulated winter substitute'
Latency: 5.4 ms
Results: 5 rows
--------------------------------------------------------------------------------
  [1] SKU-APP-04418      | Ridgeline Insulated Jacket     | score=0.600000
      Apparel > Outerwear | Insulated winter jacket, synthetic fill, water-resistant shell — a warm midweight alternative to a down parka.
  [2] SKU-APP-04412      | Summit Down Parka              | score=0.400000
      Apparel > Outerwear | Heavyweight insulated winter parka, 600-fill down, waterproof shell, storm hood — warmest cold-weather outerwear.
  [3] SKU-APP-10005      | Jacket 844                     | score=0.200000
      Apparel > Tops | all-season tops jacket in apparel.
  [4] SKU-APP-10013      | Jacket 448                     | score=0.200000
      Apparel > Tops | all-season tops jacket in apparel.
  [5] SKU-APP-04460      | Frostguard Thermal Gloves      | score=0.200000
      Apparel > Accessories | Insulated thermal winter gloves, to

In [4]:
# Verify index definition
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT indexname, indexdef FROM pg_indexes
        WHERE schemaname = 'northpeak_app' AND indexname = 'idx_product_search_bm25'
    """)
    idx = cur.fetchall()
print("Index verification:")
for i in idx:
    print(f"  Name: {i['indexname']}")
    print(f"  Def:  {i['indexdef']}")
conn.close()

Index verification:
  Name: idx_product_search_bm25
  Def:  CREATE INDEX idx_product_search_bm25 ON northpeak_app.product_search USING gin (search_vector)
